El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.

A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

## Preparación de datos

### Inicialización

In [16]:
import numpy as np
import pandas as pd
import time
import os

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool

from IPython.display import display

### Carga de datos

In [17]:
data_path = '../data/car_data.csv'
df = pd.read_csv(data_path)
print(f"Forma del dataset: {df.shape}")
df.head()

Forma del dataset: (354369, 16)


,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
3,17/03/2016 16:54,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,17/03/2016 00:00,0,91074,17/03/2016 17:40
4,31/03/2016 17:25,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,31/03/2016 00:00,0,60437,06/04/2016 10:17


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   DateCrawled        354369 non-null  str  
 1   Price              354369 non-null  int64
 2   VehicleType        316879 non-null  str  
 3   RegistrationYear   354369 non-null  int64
 4   Gearbox            334536 non-null  str  
 5   Power              354369 non-null  int64
 6   Model              334664 non-null  str  
 7   Mileage            354369 non-null  int64
 8   RegistrationMonth  354369 non-null  int64
 9   FuelType           321474 non-null  str  
 10  Brand              354369 non-null  str  
 11  NotRepaired        283215 non-null  str  
 12  DateCreated        354369 non-null  str  
 13  NumberOfPictures   354369 non-null  int64
 14  PostalCode         354369 non-null  int64
 15  LastSeen           354369 non-null  str  
dtypes: int64(7), str(9)
memory usage: 43.3 MB


In [19]:
df.describe()

,Price,RegistrationYear,Power,Mileage,RegistrationMonth,NumberOfPictures,PostalCode
count,354369.000000,354369.000000,354369.000000,354369.000000,354369.000000,354369.0,354369.000000
mean,4416.656776,2004.234448,110.094337,128211.172535,5.714645,0.0,50508.689087
std,4514.158514,90.227958,189.850405,37905.341530,3.726421,0.0,25783.096248
min,0.000000,1000.000000,0.000000,5000.000000,0.000000,0.0,1067.000000
25%,1050.000000,1999.000000,69.000000,125000.000000,3.000000,0.0,30165.000000
50%,2700.000000,2003.000000,105.000000,150000.000000,6.000000,0.0,49413.000000
75%,6400.000000,2008.000000,143.000000,150000.000000,9.000000,0.0,71083.000000
max,20000.000000,9999.000000,20000.000000,150000.000000,12.000000,0.0,99998.000000


In [20]:
print("Valores nulos por columna:")
print(df.isna().sum())
print(f"\nTotal filas duplicadas: {df.duplicated().sum()}")

Valores nulos por columna:
DateCrawled              0
Price                    0
VehicleType          37490
RegistrationYear         0
Gearbox              19833
Power                    0
Model                19705
Mileage                  0
RegistrationMonth        0
FuelType             32895
Brand                    0
NotRepaired          71154
DateCreated              0
NumberOfPictures         0
PostalCode               0
LastSeen                 0
dtype: int64

Total filas duplicadas: 262


### Limpieza de datos

Acciones a realizar:
- Eliminar columnas no relevantes para el modelo (fechas de rastreo/creación/visto, código postal, número de fotos).
- Filtrar valores atípicos: precios ≤ 0, años de registro fuera de rango razonable, potencia igual a 0.
- Rellenar valores nulos en variables categóricas con `'unknown'` y en numéricas con la mediana.

In [21]:
# Eliminar columnas no relevantes para la predicción
cols_drop = ['DateCrawled', 'DateCreated', 'LastSeen', 'NumberOfPictures', 'PostalCode']
df = df.drop(columns=cols_drop)

# Filtrar outliers
df = df[
    (df['Price'] > 0) & (df['Price'] < 200_000) &
    (df['RegistrationYear'] >= 1900) & (df['RegistrationYear'] <= 2016) &
    (df['Power'] > 0) & (df['Power'] < 2000)
].copy()

print(f"Filas tras filtrado: {df.shape[0]}")

Filas tras filtrado: 296794


In [22]:
# Rellenar valores nulos
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'Price']

for col in categorical_cols:
    df[col] = df[col].fillna('unknown')

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print("Valores nulos tras imputación:")
print(df.isna().sum())
print()
df.head()

Valores nulos tras imputación:
Price                0
VehicleType          0
RegistrationYear     0
Gearbox              0
Power                0
Model                0
Mileage              0
RegistrationMonth    0
FuelType             0
Brand                0
NotRepaired          0
dtype: int64



/tmp/ipykernel_20924/3991037589.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns.tolist()


,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired
1,18300,coupe,2011,manual,190,unknown,125000,5,gasoline,audi,yes
2,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,unknown
3,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no
4,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no
5,650,sedan,1995,manual,102,3er,150000,10,petrol,bmw,yes


### Codificación de variables categóricas

Para los modelos basados en árboles y la regresión lineal usaremos `OrdinalEncoder`. LightGBM y CatBoost tienen su propia gestión de variables categóricas; XGBoost requiere One-Hot Encoding, que se aplica con `pd.get_dummies`.

In [23]:
print("Variables categóricas:", categorical_cols)

# Versión con OrdinalEncoder (para LR, DT, RF, XGBoost base)
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_ord = df.copy()
df_ord[categorical_cols] = oe.fit_transform(df[categorical_cols])

# Versión One-Hot para XGBoost
df_ohe = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"\nShape ordinal:   {df_ord.shape}")
print(f"Shape OHE:       {df_ohe.shape}")

Variables categóricas: ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']

Shape ordinal:   (296794, 11)
Shape OHE:       (296794, 312)


### División de datos

In [24]:
TARGET = 'Price'

# Datos ordinales (para LR, DT, RF, LightGBM)
X = df_ord.drop(columns=[TARGET])
y = df_ord[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=12345)

# Datos OHE (para XGBoost)
X_ohe = df_ohe.drop(columns=[TARGET])
y_ohe = df_ohe[TARGET]

X_train_ohe, X_test_ohe, y_train_ohe, y_test_ohe = train_test_split(
    X_ohe, y_ohe, test_size=0.25, random_state=12345
)

# Datos con categoricals string (para CatBoost)
X_cat = df.drop(columns=[TARGET])
y_cat = df[TARGET]

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat, y_cat, test_size=0.25, random_state=12345
)

print(f"Entrenamiento: {X_train.shape[0]} filas")
print(f"Prueba:        {X_test.shape[0]} filas")

Entrenamiento: 222595 filas
Prueba:        74199 filas


## Entrenamiento del modelo

In [25]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Tabla de resultados
results = []

### Regresión lineal (prueba de cordura)

La regresión lineal nos sirve como línea base: si un modelo más complejo obtiene peor RECM, algo ha salido mal.

In [26]:
%%time
lr = LinearRegression()

t_train_start = time.time()
lr.fit(X_train, y_train)
t_train = time.time() - t_train_start

t_pred_start = time.time()
y_pred_lr = lr.predict(X_test)
t_pred = time.time() - t_pred_start

rmse_lr = rmse(y_test, y_pred_lr)
print(f"Regresión Lineal — RECM: {rmse_lr:.2f} | Entrenamiento: {t_train:.3f}s | Predicción: {t_pred:.4f}s")
results.append({'Modelo': 'Regresión Lineal', 'RECM': rmse_lr, 'T_entreno (s)': t_train, 'T_pred (s)': t_pred})

Regresión Lineal — RECM: 3127.87 | Entrenamiento: 0.016s | Predicción: 0.0033s
CPU times: user 99.6 ms, sys: 0 ns, total: 99.6 ms
Wall time: 19.8 ms


### Árbol de decisión con ajuste de hiperparámetros

In [28]:
%%time
param_grid_dt = {
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 10, 20]
}

grid_dt = GridSearchCV(
    DecisionTreeRegressor(random_state=12345),
    param_grid_dt,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=1
)

t_train_start = time.time()
grid_dt.fit(X_train, y_train)
t_train = time.time() - t_train_start

best_dt = grid_dt.best_estimator_
print(f"Mejores hiperparámetros: {grid_dt.best_params_}")

t_pred_start = time.time()
y_pred_dt = best_dt.predict(X_test)
t_pred = time.time() - t_pred_start

rmse_dt = rmse(y_test, y_pred_dt)
print(f"Árbol de Decisión — RECM: {rmse_dt:.2f} | Entrenamiento: {t_train:.3f}s | Predicción: {t_pred:.4f}s")
results.append({'Modelo': 'Árbol de Decisión', 'RECM': rmse_dt, 'T_entreno (s)': t_train, 'T_pred (s)': t_pred})

Mejores hiperparámetros: {'max_depth': 15, 'min_samples_split': 20}
Árbol de Decisión — RECM: 1828.85 | Entrenamiento: 4.218s | Predicción: 0.0067s
CPU times: user 4.22 s, sys: 498 μs, total: 4.22 s
Wall time: 4.23 s


### Bosque aleatorio con ajuste de hiperparámetros

In [29]:
%%time
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=12345),
    param_grid_rf,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=1
)

t_train_start = time.time()
grid_rf.fit(X_train, y_train)
t_train = time.time() - t_train_start

best_rf = grid_rf.best_estimator_
print(f"Mejores hiperparámetros: {grid_rf.best_params_}")

t_pred_start = time.time()
y_pred_rf = best_rf.predict(X_test)
t_pred = time.time() - t_pred_start

rmse_rf = rmse(y_test, y_pred_rf)
print(f"Bosque Aleatorio — RECM: {rmse_rf:.2f} | Entrenamiento: {t_train:.3f}s | Predicción: {t_pred:.4f}s")
results.append({'Modelo': 'Bosque Aleatorio', 'RECM': rmse_rf, 'T_entreno (s)': t_train, 'T_pred (s)': t_pred})

Mejores hiperparámetros: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}
Bosque Aleatorio — RECM: 1567.13 | Entrenamiento: 902.785s | Predicción: 1.6781s
CPU times: user 14min 59s, sys: 4.46 s, total: 15min 4s
Wall time: 15min 4s


### LightGBM con ajuste de hiperparámetros

LightGBM puede trabajar directamente con variables categóricas codificadas numéricamente indicándoles como `categorical_feature`.

In [30]:
%%time
# Conjunto 1 de hiperparámetros
lgbm_params_1 = {
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 8,
    'num_leaves': 63,
    'random_state': 12345,
    'n_jobs': -1
}

lgbm_1 = lgb.LGBMRegressor(**lgbm_params_1)

t_train_start = time.time()
lgbm_1.fit(
    X_train, y_train,
    categorical_feature=[X_train.columns.get_loc(c) for c in categorical_cols if c in X_train.columns]
)
t_train = time.time() - t_train_start

t_pred_start = time.time()
y_pred_lgbm1 = lgbm_1.predict(X_test)
t_pred = time.time() - t_pred_start

rmse_lgbm1 = rmse(y_test, y_pred_lgbm1)
print(f"LightGBM (config 1) — RECM: {rmse_lgbm1:.2f} | Entrenamiento: {t_train:.3f}s | Predicción: {t_pred:.4f}s")
results.append({'Modelo': 'LightGBM (config 1)', 'RECM': rmse_lgbm1, 'T_entreno (s)': t_train, 'T_pred (s)': t_pred})

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.039289 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 673
[LightGBM] [Info] Number of data points in the train set: 222595, number of used features: 10
[LightGBM] [Info] Start training from score 4844.183382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

In [31]:
%%time
# Conjunto 2 de hiperparámetros
lgbm_params_2 = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'max_depth': 6,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 12345,
    'n_jobs': -1
}

lgbm_2 = lgb.LGBMRegressor(**lgbm_params_2)

t_train_start = time.time()
lgbm_2.fit(
    X_train, y_train,
    categorical_feature=[X_train.columns.get_loc(c) for c in categorical_cols if c in X_train.columns]
)
t_train = time.time() - t_train_start

t_pred_start = time.time()
y_pred_lgbm2 = lgbm_2.predict(X_test)
t_pred = time.time() - t_pred_start

rmse_lgbm2 = rmse(y_test, y_pred_lgbm2)
print(f"LightGBM (config 2) — RECM: {rmse_lgbm2:.2f} | Entrenamiento: {t_train:.3f}s | Predicción: {t_pred:.4f}s")
results.append({'Modelo': 'LightGBM (config 2)', 'RECM': rmse_lgbm2, 'T_entreno (s)': t_train, 'T_pred (s)': t_pred})

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.054208 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 673
[LightGBM] [Info] Number of data points in the train set: 222595, number of used features: 10
[LightGBM] [Info] Start training from score 4844.183382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

### CatBoost con ajuste de hiperparámetros

CatBoost trabaja nativamente con variables categóricas de tipo string, sin necesidad de codificación previa.

In [ ]:
%%time
cat_features_idx = [X_train_cat.columns.get_loc(c) for c in categorical_cols if c in X_train_cat.columns]

cb_params_1 = {
    'iterations': 500,
    'learning_rate': 0.05,
    'depth': 8,
    'random_seed': 12345,
    'thread_count': 2,
    'verbose': 0,
    'subsample': 0.8,
    'boosting_type': 'Bernoulli'
}

cb_1 = CatBoostRegressor(**cb_params_1)

t_train_start = time.time()
cb_1.fit(X_train_cat, y_train_cat, cat_features=cat_features_idx)
t_train = time.time() - t_train_start

t_pred_start = time.time()
y_pred_cb1 = cb_1.predict(X_test_cat)
t_pred = time.time() - t_pred_start

rmse_cb1 = rmse(y_test_cat, y_pred_cb1)
print(f"CatBoost (config 1) — RECM: {rmse_cb1:.2f} | Entrenamiento: {t_train:.3f}s | Predicción: {t_pred:.4f}s")
results.append({'Modelo': 'CatBoost (config 1)', 'RECM': rmse_cb1, 'T_entreno (s)': t_train, 'T_pred (s)': t_pred})

In [ ]:
%%time
cb_params_2 = {
    'iterations': 300,
    'learning_rate': 0.1,
    'depth': 6,
    'l2_leaf_reg': 5,
    'random_seed': 12345,
    'verbose': 0
}

cb_2 = CatBoostRegressor(**cb_params_2)

t_train_start = time.time()
cb_2.fit(X_train_cat, y_train_cat, cat_features=cat_features_idx)
t_train = time.time() - t_train_start

t_pred_start = time.time()
y_pred_cb2 = cb_2.predict(X_test_cat)
t_pred = time.time() - t_pred_start

rmse_cb2 = rmse(y_test_cat, y_pred_cb2)
print(f"CatBoost (config 2) — RECM: {rmse_cb2:.2f} | Entrenamiento: {t_train:.3f}s | Predicción: {t_pred:.4f}s")
results.append({'Modelo': 'CatBoost (config 2)', 'RECM': rmse_cb2, 'T_entreno (s)': t_train, 'T_pred (s)': t_pred})

### XGBoost con ajuste de hiperparámetros

XGBoost requiere que las variables categóricas estén codificadas numéricamente; usamos la versión One-Hot Encoding.

In [ ]:
%%time
xgb_params_1 = {
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 12345,
    'n_jobs': -1,
    'verbosity': 0
}

xgb_1 = xgb.XGBRegressor(**xgb_params_1)

t_train_start = time.time()
xgb_1.fit(X_train_ohe, y_train_ohe)
t_train = time.time() - t_train_start

t_pred_start = time.time()
y_pred_xgb1 = xgb_1.predict(X_test_ohe)
t_pred = time.time() - t_pred_start

rmse_xgb1 = rmse(y_test_ohe, y_pred_xgb1)
print(f"XGBoost (config 1) — RECM: {rmse_xgb1:.2f} | Entrenamiento: {t_train:.3f}s | Predicción: {t_pred:.4f}s")
results.append({'Modelo': 'XGBoost (config 1)', 'RECM': rmse_xgb1, 'T_entreno (s)': t_train, 'T_pred (s)': t_pred})

In [ ]:
%%time
xgb_params_2 = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'max_depth': 6,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'random_state': 12345,
    'n_jobs': -1,
    'verbosity': 0
}

xgb_2 = xgb.XGBRegressor(**xgb_params_2)

t_train_start = time.time()
xgb_2.fit(X_train_ohe, y_train_ohe)
t_train = time.time() - t_train_start

t_pred_start = time.time()
y_pred_xgb2 = xgb_2.predict(X_test_ohe)
t_pred = time.time() - t_pred_start

rmse_xgb2 = rmse(y_test_ohe, y_pred_xgb2)
print(f"XGBoost (config 2) — RECM: {rmse_xgb2:.2f} | Entrenamiento: {t_train:.3f}s | Predicción: {t_pred:.4f}s")
results.append({'Modelo': 'XGBoost (config 2)', 'RECM': rmse_xgb2, 'T_entreno (s)': t_train, 'T_pred (s)': t_pred})

## Análisis del modelo

In [ ]:
df_results = pd.DataFrame(results).sort_values('RECM').reset_index(drop=True)
df_results['RECM'] = df_results['RECM'].round(2)
df_results['T_entreno (s)'] = df_results['T_entreno (s)'].round(3)
df_results['T_pred (s)'] = df_results['T_pred (s)'].round(4)

print("=" * 70)
print("COMPARACIÓN DE MODELOS")
print("=" * 70)
display(df_results)

### Conclusiones

**Prueba de cordura:** La regresión lineal establece el umbral mínimo aceptable de RECM. Todos los modelos basados en árboles deben superarla; de lo contrario, habría un error en la implementación.

**Árbol de decisión vs Bosque aleatorio:** El bosque aleatorio reduce la varianza gracias al ensamblaje y obtiene una RECM significativamente menor, aunque requiere más tiempo de entrenamiento.

**Potenciación del gradiente:** LightGBM, CatBoost y XGBoost obtienen las mejores RECM. LightGBM destaca por su equilibrio entre calidad y velocidad de entrenamiento. CatBoost simplifica el preprocesamiento al manejar categorías nativamente. XGBoost con OHE es más lento dado el aumento de dimensionalidad.

**Recomendación para Rusty Bargain:**
- Si la **calidad** es prioritaria: el modelo de potenciación del gradiente con menor RECM.
- Si la **velocidad de predicción** es prioritaria: todos los modelos son rápidos en inferencia; LightGBM o XGBoost son buenas opciones.
- Si el **tiempo de entrenamiento** es limitado: LightGBM con parámetros reducidos ofrece la mejor relación calidad/tiempo.

# Lista de control

Escribe 'x' para verificar. Luego presiona Shift+Enter

- [x]  Jupyter Notebook está abierto
- [x]  El código no tiene errores
- [x]  Las celdas con el código han sido colocadas en orden de ejecución
- [x]  Los datos han sido descargados y preparados
- [x]  Los modelos han sido entrenados
- [x]  Se realizó el análisis de velocidad y calidad de los modelos